# Task Execution Mode

Task execution mode is a behavioral pattern where the agent receives a specific deliverable and works systematically to produce it. Success is measured objectively: did the deliverable get produced, does it meet the required quality criteria, and was it completed within defined constraints?

Task execution mode is goal-directed: there is a clear definition of done, and the agent works toward it without unnecessary exploration or deliberation. Once the task is understood, the agent plans, executes, and verifies - in that order.

This mode is classified as goal-directed completion. It can combine with any autonomy level:
- Task execution + chat: Agent explains how to do a task, but the human executes.
- Task execution + copilot: Agent drafts deliverables, human reviews and finalizes.
- Task execution + supervised: Agent completes phases, human approves at checkpoints.
- Task execution + fully autonomous: Agent completes end-to-end and reports results.

In [1]:
import os
import json
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model
Task execution mode uses `temperature=0` for deterministic, reproducible outputs. When executing a defined task, creative variation is a liability - the agent should produce the same quality deliverable consistently, not vary its interpretation of the goal between runs.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

print("LLM initialized for task execution mode.")

LLM initialized for task execution mode.


## Defining a task specification
Before the agent can execute anything, it needs a precise definition of what success looks like. The most common failure in task execution mode is starting work with an ambiguous goal - the agent finishes something, but not necessarily the right thing. Frontloading clarity through a `TaskSpec` prevents this.

A task specification bundles four things the agent needs upfront: the goal in human-readable terms, a description of the concrete artifact to produce, a list of criteria the output will be judged against, and any constraints that bound the scope. This upfront contract is what separates task execution from open-ended generation - once the spec is defined, both the agent and any human reviewer share a common definition of "done."

The `TaskResult` counterpart records what the agent actually produced, along with a per-criterion quality assessment. Keeping these as two separate structures makes it straightforward to compare what was asked for against what was delivered, which is essential when a quality score falls short and a reviewer needs to understand exactly which criteria failed.

In [3]:
@dataclass
class TaskSpec:
    """Specification for a task to be executed by the agent."""
    goal: str                    # what the agent is trying to achieve, in plain language
    deliverable: str             # concrete artifact or output to produce
    quality_criteria: list       # the bar the final output must clear, checked during verification
    constraints: dict = field(default_factory=dict)  # scope, format, or resource limits


@dataclass
class TaskResult:
    """Result produced by a completed task execution."""
    deliverable: str             # the actual output produced by the agent
    steps_taken: list            # ordered record of each step name and its output
    criteria_scores: dict        # per-criterion quality score, from 0.0 to 1.0
    quality_score: float         # aggregate score averaged across all per-criterion scores
    quality_notes: str           # explanation of scores and any remaining gaps
    status: str                  # 'completed' when score >= 0.7, otherwise 'partial'

Both classes serve distinct roles in the pipeline. `TaskSpec` is the *input contract* - it is written once before execution begins and never modified during the run. `TaskResult` is the *output record* - it captures the deliverable alongside the evidence used to evaluate it, making the agent's work auditable.

The separation between `criteria_scores` and `quality_score` is deliberate. The per-criterion breakdown tells a reviewer exactly which criteria passed and which fell short, while the aggregate drives automated pass/fail decisions. Collapsing failures into a passing average would hide the specific problems a reviewer needs to address.

In [4]:
# Define the task we will use throughout this notebook — a FastAPI health-check module
task = TaskSpec(
    goal="Write a Python module that exposes a REST health-check endpoint",
    deliverable="A Python file using FastAPI with a GET /health endpoint returning JSON status",
    quality_criteria=[
        "Contains a GET /health endpoint",
        "Returns JSON with at least 'status' and 'timestamp' fields",
        "Uses FastAPI framework correctly",
        "Includes module-level docstring and type hints",
        "Is runnable as a standalone script",
    ],
    constraints={"max_lines": 50, "framework": "FastAPI", "python_version": "3.9+"}
)

# Inspect the spec before any execution — this is the shared definition of 'done'
print(f"Goal        : {task.goal}")
print(f"Deliverable : {task.deliverable}")
print(f"\nQuality criteria ({len(task.quality_criteria)}):")
for criterion in task.quality_criteria:
    print(f"  - {criterion}")
print(f"\nConstraints : {task.constraints}")

Goal        : Write a Python module that exposes a REST health-check endpoint
Deliverable : A Python file using FastAPI with a GET /health endpoint returning JSON status

Quality criteria (5):
  - Contains a GET /health endpoint
  - Returns JSON with at least 'status' and 'timestamp' fields
  - Uses FastAPI framework correctly
  - Includes module-level docstring and type hints
  - Is runnable as a standalone script

Constraints : {'max_lines': 50, 'framework': 'FastAPI', 'python_version': '3.9+'}


## Step 1: Planning the task
The first move in task execution is decomposition: the agent translates the goal into an ordered sequence of concrete, verifiable steps. Planning happens once, before any execution begins, and the resulting plan is treated as a fixed contract for the rest of the run.

Why plan first rather than improvise step by step? Improvised execution tends to drift - the agent may realize mid-task that it chose the wrong approach and must backtrack, wasting earlier work. A plan created upfront makes the full intended sequence visible before the first step is taken, giving any human reviewer a chance to redirect before effort is spent.

Each step in the plan carries an identifier, a short name, a description of what to do, and an optional tool name. Keeping this schema minimal ensures the planning step stays focused on *what* to do in what order - the details of *how* each step executes are left to the execution phase.

In [5]:
def plan_task(spec: TaskSpec) -> list:
    """Decompose a task specification into an ordered list of execution steps.

    Args:
        spec: The task specification with goal, deliverable, and criteria.

    Returns:
        List of step dicts, each with 'id', 'name', 'description', and 'tool'.
    """
    system_prompt = """You are a task planner. Break the given task into 3-5 concrete, ordered steps.

Available tools:
- write_file: Write content to a file (use when the deliverable is a file)
- null: Pure reasoning or LLM-generated content step (no tool needed)

Respond with JSON only:
{"steps": [{"id": "step_1", "name": "short name", "description": "what to do", "tool": "write_file or null"}]}"""

    # Include quality criteria and constraints so the planner knows the full bar to clear
    prompt = f"""Goal: {spec.goal}
Deliverable: {spec.deliverable}
Quality criteria: {spec.quality_criteria}
Constraints: {spec.constraints}"""

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        # Parse the JSON plan — asking for JSON directly avoids natural-language parsing
        data = json.loads(response.content)
        return data["steps"]
    except (json.JSONDecodeError, KeyError):
        # Fall back to a single generic step so the execution layer always has something to run
        return [{"id": "step_1", "name": "Execute task", "description": spec.goal, "tool": None}]

The function asks the LLM to produce JSON rather than prose, which keeps the plan machine-readable without any natural-language parsing on our side. The try/except fallback to a single-step plan ensures the pipeline can always proceed - graceful degradation rather than a hard crash.

Passing the quality criteria and constraints into the planning prompt is a deliberate choice. A plan generated without awareness of the quality bar tends to skip steps that only become necessary during verification - for instance, adding type hints or a module docstring. Including the full spec upfront gives the planner everything it needs to produce a plan that will actually pass verification.

In [6]:
# Run just the planning step to see the proposed approach before any code is written
steps = plan_task(task)

print(f"Plan: {len(steps)} steps")
print("-" * 50)
for s in steps:
    # Flag which steps will call a tool so we can see the execution shape upfront
    tool_label = f" [tool: {s['tool']}]" if s.get("tool") and s["tool"] != "null" else ""
    print(f"  {s['id']}: {s['name']}{tool_label}")
    print(f"           {s['description']}")

Plan: 5 steps
--------------------------------------------------
  step_1: Create Python file [tool: write_file]
           Create a new Python file for the FastAPI module.
  step_2: Add module-level docstring
           Include a module-level docstring explaining the purpose of the module.
  step_3: Define health-check endpoint
           Implement the GET /health endpoint that returns JSON with 'status' and 'timestamp'.
  step_4: Add type hints
           Ensure all functions and parameters have appropriate type hints.
  step_5: Make it runnable
           Add the necessary code to allow the script to be run as a standalone application.


## Step 2: Executing the plan
With a plan in hand, the agent executes each step sequentially. The key design decision here is context accumulation: the output of each completed step is appended to a running context string, which is included in the prompt for every subsequent step. This means later steps are always informed by what earlier steps produced - no step executes in isolation.

One mock tool is available in this notebook: `write_file`, which simulates writing output to disk. A tool registry (`TOOLS` dict) maps tool names from the plan to callable Python functions. This indirection decouples the execution loop from any specific tool implementation - replacing the mock with a real filesystem write only requires changing the registry, not the loop.

The execution logic follows a single unified path for every step: the LLM generates content, and if the step specifies a tool, that tool is called with the generated content. There is no separate code path for tool-using steps versus reasoning-only steps - the difference is simply whether the tool call happens after the LLM response.

In [7]:
def write_file(filename: str, content: str) -> str:
    """Simulate writing content to a file.

    In production, replace the body with a real filesystem or cloud storage write.
    """
    # Mock: return a confirmation message instead of touching the filesystem
    return f"[mock] Wrote {len(content)} characters to '{filename}'."


# Tool registry: maps plan tool names to callables — swap mocks for real implementations here
TOOLS = {"write_file": write_file}


def execute_task(spec: TaskSpec, steps: list) -> tuple:
    """Execute the planned steps sequentially and collect results.

    Args:
        spec: The original task specification (goal, deliverable, constraints).
        steps: The ordered list of steps produced by plan_task().

    Returns:
        Tuple of (final_deliverable: str, steps_taken: list).
    """
    steps_taken = []
    accumulated_context = ""  # grows with each completed step; passed into every subsequent prompt
    final_output = ""         # set once a step produces the main deliverable

    for step in steps:
        # Include everything the agent has produced so far so later steps can build on earlier ones
        prompt = f"""Goal: {spec.goal}
Deliverable: {spec.deliverable}
Current step: {step['description']}
Context from completed steps: {accumulated_context or 'None yet'}

Complete this step thoroughly."""

        resp = llm.invoke([
            SystemMessage(content="You are a precise task executor. Complete the described step."),
            HumanMessage(content=prompt)
        ])
        step_output = resp.content.strip()

        # If this step uses a tool, call it now with the content the LLM just produced
        tool_name = step.get("tool")
        tool_result = ""
        if tool_name and tool_name in TOOLS:
            tool_result = TOOLS[tool_name]("output.py", step_output)
            final_output = step_output   # the content written to the file IS the deliverable

        # Record this step's result — the full trace feeds into verification and human review
        steps_taken.append({
            "id": step["id"],
            "name": step["name"],
            "output": step_output,
            "tool_result": tool_result,
            "status": "done"
        })

        # Append a short summary so the next step can reference what was done here
        accumulated_context += f"\n[{step['name']}]: {step_output[:300]}"

    # If no write_file step produced a file, synthesize the deliverable from all accumulated work
    if not final_output:
        synth_resp = llm.invoke([
            SystemMessage(content="Synthesize the final deliverable from the completed work below."),
            HumanMessage(content=f"Goal: {spec.goal}\nDeliverable: {spec.deliverable}\n\nCompleted work:\n{accumulated_context}")
        ])
        final_output = synth_resp.content.strip()

    return final_output, steps_taken

The loop structure reflects a single execution pattern regardless of whether a step uses a tool. The LLM always generates content first; the tool call, if any, is a side-effect that happens afterward. This uniformity makes the loop easy to extend - introducing a new tool requires one entry in `TOOLS` and nothing else changes.

The `accumulated_context` string is truncated to 300 characters per step before being appended. Without this truncation, a long chain of steps could push the context window past its limit. The truncation is lossy by design: it preserves the shape of what was done without carrying every detail forward, which is usually sufficient for subsequent steps to build on.

The fallback synthesis at the end handles the case where all steps were reasoning-only. Rather than returning an empty string, the agent makes one final LLM call to consolidate the accumulated work into the requested deliverable form.

In [8]:
# Execute the plan produced in the previous step — only plan_task needs to run first
deliverable, steps_taken = execute_task(task, steps)

print(f"Executed {len(steps_taken)} steps:")
print("-" * 50)
for s in steps_taken:
    # Truncate long outputs for display — full output is in steps_taken for later use
    preview = s["output"][:80] + "..." if len(s["output"]) > 80 else s["output"]
    print(f"  [DONE] {s['name']}")
    print(f"         {preview}")
    if s["tool_result"]:
        # Show the tool confirmation when a tool was invoked on this step
        print(f"         {s['tool_result']}")

print(f"\nDeliverable preview (first 300 chars):")
print("-" * 50)
print(deliverable[:300])

Executed 5 steps:
--------------------------------------------------
  [DONE] Create Python file
         To create a new Python file for the FastAPI module that exposes a REST health-ch...
         [mock] Wrote 1802 characters to 'output.py'.
  [DONE] Add module-level docstring
         Here is the Python file `health_check.py` with a module-level docstring explaini...
  [DONE] Define health-check endpoint
         Here is the implementation of the GET `/health` endpoint in the `health_check.py...
  [DONE] Add type hints
         Here is the complete `health_check.py` file with appropriate type hints for all ...
  [DONE] Make it runnable
         To allow the `health_check.py` script to be run as a standalone application, you...

Deliverable preview (first 300 chars):
--------------------------------------------------
To create a new Python file for the FastAPI module that exposes a REST health-check endpoint, follow these steps:

1. **Create a new Python file**: You can name the file

## Step 3: Verifying quality
Producing a deliverable and producing a *good* deliverable are different things. The verification step closes this gap: the agent evaluates its own output against each criterion in the task spec, assigns a score per criterion, and computes an aggregate quality score.

This self-assessment is the checkpoint that prevents the agent from treating "I produced something" as equivalent to "I produced something that meets the spec." Without it, the pipeline has no principled way to distinguish a high-quality output from one that superficially looks complete but fails important criteria. The per-criterion breakdown also gives the human reviewer precise, actionable information - not just a score, but which specific criteria need more work.

The quality threshold of 0.7 is the dividing line between a `completed` result and a `partial` one. A `partial` status is not a failure - it is an honest signal that the deliverable exists but did not fully clear the bar, and that a human should review before using it.

In [9]:
def verify_quality(spec: TaskSpec, deliverable: str) -> dict:
    """Evaluate the deliverable against each quality criterion in the task spec.

    Args:
        spec: The task specification containing quality_criteria.
        deliverable: The produced output to evaluate.

    Returns:
        Dict with 'criteria_scores' (per-criterion 0.0-1.0), 'overall_score', and 'notes'.
    """
    # Format criteria as a bulleted list so the reviewer prompt is easy for the LLM to parse
    criteria_formatted = "\n".join(
        f"- {criterion}" for criterion in spec.quality_criteria
    )

    system_prompt = """You are a quality reviewer. Evaluate the given output against each criterion.
Respond with JSON only — no markdown fences:
{"criteria_scores": {"criterion text": score_0_to_1}, "notes": "brief explanation of scores and any gaps"}"""

    prompt = f"""Task goal: {spec.goal}
Expected deliverable: {spec.deliverable}

Quality criteria:
{criteria_formatted}

Output to evaluate:
{deliverable[:2000]}"""   # truncate to avoid hitting context limits on very long outputs

    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    try:
        data = json.loads(response.content)
        scores = data.get("criteria_scores", {})
        # Average all per-criterion scores to produce a single overall figure
        overall = sum(scores.values()) / len(scores) if scores else 0.0
        return {
            "criteria_scores": scores,
            "overall_score": round(overall, 2),
            "notes": data.get("notes", "")
        }
    except (json.JSONDecodeError, KeyError):
        # Return a zero score rather than crashing — the pipeline can still report partial status
        return {"criteria_scores": {}, "overall_score": 0.0, "notes": "Quality evaluation failed."}

The function asks the reviewer to return JSON keyed by the criterion text, which ties each score back to the exact wording in the `TaskSpec`. That linkage matters for transparency: a score of `0.4` on "Uses FastAPI framework correctly" is immediately actionable, whereas a score of `0.4` on an unlabeled dimension is not.

The deliverable is truncated to 2000 characters before being included in the prompt. This is a pragmatic limit - most quality criteria can be assessed from the leading portion of a code file or document, and avoiding context overflows is more important than evaluating every line. In a production system, this limit could be tuned per task type or the deliverable could be chunked and evaluated in parts.

The fallback on parsing failure returns a zero overall score, which means the pipeline will always report `partial` status rather than silently calling a failed verification a success.

In [10]:
# Run quality verification on the deliverable produced in the previous step
quality = verify_quality(task, deliverable)

print(f"Overall quality score: {quality['overall_score']:.2f} / 1.00")
print("-" * 50)
print("Per-criterion scores:")
for criterion, score in quality["criteria_scores"].items():
    # Show a checkmark for passing criteria (>= 0.7) and an X for those that need work
    icon = "\u2713" if score >= 0.7 else "\u2717"
    print(f"  {icon} {criterion}: {score:.2f}")
print(f"\nNotes: {quality['notes']}")

Overall quality score: 0.60 / 1.00
--------------------------------------------------
Per-criterion scores:
  ✓ Contains a GET /health endpoint: 1.00
  ✗ Returns JSON with at least 'status' and 'timestamp' fields: 0.00
  ✓ Uses FastAPI framework correctly: 1.00
  ✗ Includes module-level docstring and type hints: 0.00
  ✓ Is runnable as a standalone script: 1.00

Notes: The output correctly implements a GET /health endpoint and uses FastAPI properly. However, it only returns the 'status' field in the JSON response and lacks the 'timestamp' field. Additionally, there is no module-level docstring or type hints provided, which are necessary for clarity and type safety.


## Complete task execution pipeline
The three steps - plan, execute, verify - now assemble into a single `run_task` function. This is the agent's public interface: callers provide a `TaskSpec` and receive a `TaskResult`. The pipeline logs its progress at each stage so the run is transparent to whoever is watching.

The pass/fail threshold is set at 0.7. Outputs that meet this threshold are marked `completed`; those that fall short are marked `partial`, signaling that the deliverable exists but needs review before use. Reporting `partial` honestly is more valuable than rounding up to `completed` - it tells the downstream consumer not to treat the output as final.

In [11]:
def run_task(spec: TaskSpec) -> TaskResult:
    """Execute a task end-to-end: plan, execute, and verify quality.

    Args:
        spec: The complete task specification.

    Returns:
        A TaskResult with the deliverable, steps taken, and quality assessment.
    """
    print(f"\nTASK: {spec.goal}")
    print("-" * 60)

    # --- Step 1: Plan -------------------------------------------------------
    print("\n[1/3] Planning...")
    steps = plan_task(spec)
    for s in steps:
        # Annotate tool-using steps in the progress log so the shape of the run is clear
        tool_label = f" [tool: {s['tool']}]" if s.get("tool") and s["tool"] != "null" else ""
        print(f"  {s['id']}: {s['name']}{tool_label}")
        print(f"           {s['description']}")

    # --- Step 2: Execute ----------------------------------------------------
    print("\n[2/3] Executing...")
    deliverable, steps_taken = execute_task(spec, steps)
    for s in steps_taken:
        # Truncate output previews so the progress log stays readable
        preview = s["output"][:80] + "..." if len(s["output"]) > 80 else s["output"]
        print(f"  [DONE] {s['name']}: {preview}")

    # --- Step 3: Verify -----------------------------------------------------
    print("\n[3/3] Verifying quality...")
    quality = verify_quality(spec, deliverable)

    # Determine status based on whether the aggregate score clears the pass threshold
    status = "completed" if quality["overall_score"] >= 0.7 else "partial"

    result = TaskResult(
        deliverable=deliverable,
        steps_taken=steps_taken,
        criteria_scores=quality["criteria_scores"],
        quality_score=quality["overall_score"],
        quality_notes=quality["notes"],
        status=status
    )

    # Print a summary with per-criterion evidence so a reviewer can accept or reject quickly
    print(f"\n{'='*60}")
    print("TASK COMPLETE")
    print(f"{'='*60}")
    print(f"Status         : {result.status.upper()}")
    print(f"Quality score  : {result.quality_score:.2f} / 1.00")
    print("\nCriteria scores:")
    for criterion, score in result.criteria_scores.items():
        icon = "\u2713" if score >= 0.7 else "\u2717"
        print(f"  {icon} {criterion}: {score:.2f}")
    print(f"\nNotes: {result.quality_notes}")

    return result

In [12]:
# Run the complete pipeline end-to-end on the task spec defined at the top of the notebook
result = run_task(task)

print("\n" + "=" * 60)
print("DELIVERABLE PREVIEW (first 500 chars):")
print("=" * 60)
print(result.deliverable[:500])


TASK: Write a Python module that exposes a REST health-check endpoint
------------------------------------------------------------

[1/3] Planning...
  step_1: Create Python file [tool: write_file]
           Create a new Python file for the FastAPI module.
  step_2: Add module-level docstring
           Include a module-level docstring that describes the purpose of the module.
  step_3: Define health-check endpoint
           Implement the GET /health endpoint that returns JSON with 'status' and 'timestamp'.
  step_4: Add type hints
           Ensure all functions and parameters have appropriate type hints.
  step_5: Make it runnable
           Add the necessary code to make the script runnable as a standalone application.

[2/3] Executing...
  [DONE] Create Python file: To create a new Python file for the FastAPI module that exposes a REST health-ch...
  [DONE] Add module-level docstring: Here is the module-level docstring for the `health_check.py` file that describes...
  [DONE] De

## When to use task execution mode
Task execution mode is the right choice whenever the task has a clear, bounded deliverable and success can be defined in advance. The ability to write quality criteria *before* the task runs is the practical signal - if we cannot specify what "good" looks like upfront, use conversational mode first to clarify the requirements, then switch to task execution once the deliverable is well-defined.

| Scenario | Why Task Execution Mode Fits |
|----------|------------------------------|
| Code generation with a spec | Deliverable is well-defined; quality criteria are checkable |
| Document generation | Template and structure are known; output is verifiable against a rubric |
| Data transformation | Input/output schema is fixed; correctness is objectively measurable |
| Configuration generation | Constraints are explicit; output can be validated programmatically |
| Report compilation | Structure is predetermined; quality is measurable criterion by criterion |

**When NOT to use task execution mode:** if the goal is ambiguous, if the right deliverable format is unclear, or if the task requires exploring alternatives before committing to one approach — use conversational mode to clarify first, then switch to task execution.

- **Task execution mode is defined by a deliverable**: There is a concrete artifact to produce, a termination condition, and quality criteria to assess against. Without these, it is not task execution mode - it is open-ended generation.
- **Plan before acting**: Decomposing the goal into a sequence of verifiable steps before executing prevents improvisation mid-task and makes the execution auditable. A plan also gives human reviewers a chance to redirect before effort is spent.
- **Context accumulation keeps steps coherent**: Passing the output of each completed step into the prompt of the next ensures the agent builds progressively rather than executing each step in isolation.